# Verificação de repetição da Bronze

PREPARADO; ainda não executado no Fabric. Anexe o mesmo Lakehouse padrão do notebook 01.

1. Execute com p_mode = before para preservar a versão Delta inicial.
2. Reexecute o notebook 01, com o mesmo landing ZIP e p_run_id vazio.
3. Volte aqui, altere apenas p_mode para after e execute novamente.

A comparação usa time travel e EXCEPT ALL nas 16 colunas originais, preservando a multiplicidade. run_id e horário de ingestão podem mudar. Não execute VACUUM durante este teste. Execuções seriais; este teste verifica a repetição da mesma versão de origem, não republicação nem atomicidade entre camadas.


In [ ]:
p_mode = "before"  # Depois da repetição do notebook 01, mudar para "after".
p_check_id = "anp_2026_1_primeira_repeticao"
p_source_sha256 = "a2c95e5dfa324a9d7253d4e8d53022e0c08dcc5eb6f18551527ee9f29305c54c"


In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path
import re
import uuid
from pyspark.sql import functions as F

if p_mode not in {"before", "after"}:
    raise ValueError("p_mode deve ser before ou after")
if not re.fullmatch(r"[A-Za-z0-9_-]+", p_check_id):
    raise ValueError("p_check_id inválido")
if not re.fullmatch(r"[0-9a-f]{64}", p_source_sha256):
    raise ValueError("Hash inválido")
columns = ["regiao_sigla", "uf", "municipio", "revenda", "cnpj_revenda", "nome_rua",
           "numero_rua", "complemento", "bairro", "cep", "produto", "data_coleta",
           "valor_venda", "valor_compra", "unidade_medida", "bandeira"]
directory = Path("/lakehouse/default/Files/audit/bronze_replay") / p_check_id
directory.mkdir(parents=True, exist_ok=True)
baseline_path = directory / "baseline.json"

def read_version(version):
    return (spark.sql(f"SELECT * FROM bronze_anp VERSION AS OF {int(version)}")
            .where(F.col("source_sha256") == p_source_sha256).select(*columns))

latest_version = int(spark.sql("DESCRIBE HISTORY bronze_anp").agg(F.max("version")).first()[0])
if p_mode == "before":
    if baseline_path.exists():
        baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
        if baseline["source_sha256"] != p_source_sha256:
            raise ValueError("Baseline pertence a outro escopo")
    else:
        count = read_version(latest_version).count()
        if count != 422418:
            raise ValueError(f"Contagem inicial inesperada: {count}")
        baseline = {"status": "baseline_captured", "environment": "fabric",
                    "check_id": p_check_id, "source_sha256": p_source_sha256,
                    "delta_version": latest_version, "rows": count, "columns": columns,
                    "captured_at_utc": datetime.now(timezone.utc).isoformat()}
        baseline_path.write_text(json.dumps(baseline, indent=2), encoding="utf-8")
    print(json.dumps(baseline))
else:
    baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
    if baseline["source_sha256"] != p_source_sha256 or baseline["columns"] != columns:
        raise ValueError("Baseline pertence a outro escopo/contrato")
    if latest_version <= baseline["delta_version"]:
        raise ValueError("Ainda não há nova versão Delta; reexecute o notebook 01 primeiro")
    proof = {"check_id": p_check_id, "environment": "fabric",
             "source_sha256": p_source_sha256, "before_version": baseline["delta_version"],
             "after_version": latest_version, "check_run_id": str(uuid.uuid4()),
             "started_at_utc": datetime.now(timezone.utc).isoformat()}
    try:
        before = read_version(baseline["delta_version"])
        after = read_version(latest_version)
        before_rows, after_rows = before.count(), after.count()
        same_schema = before.schema == after.schema
        # EXCEPT ALL compara as 16 colunas e preserva multiplicidade de duplicatas.
        missing = before.exceptAll(after).limit(1).count() > 0
        extra = after.exceptAll(before).limit(1).count() > 0
        proof.update({"before_rows": before_rows, "after_rows": after_rows,
                      "same_schema": same_schema, "missing_rows_detected": missing,
                      "extra_rows_detected": extra})
        if before_rows != baseline["rows"] or before_rows != after_rows or not same_schema or missing or extra:
            raise ValueError("A repetição alterou o conteúdo de origem da Bronze")
        proof["status"] = "bronze_replay_validated"
    except Exception as exc:
        proof.update({"status": "failed", "error_type": type(exc).__name__})
        raise
    finally:
        proof["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
        (directory / (proof["check_run_id"] + ".json")).write_text(json.dumps(proof, indent=2), encoding="utf-8")
    print(json.dumps(proof))
